# NLP Assignment 3 — Build a Chatbot using Hugging Face Transformers

## Pipeline Flow
```
User Input → Tokenization → Model Processing → Response Generation → Display Output → Loop Until Exit
```

## Step 1 — Install Required Libraries

In [1]:
# Install Hugging Face Transformers and PyTorch
!pip install transformers torch --quiet


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Import Libraries

In [2]:
# Import necessary modules from Hugging Face Transformers and PyTorch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Libraries imported successfully!")

C:\Users\Zaara\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


## Step 3 — Load Pre-Trained Model and Tokenizer

In [3]:
# Define the model name — DialoGPT-medium is ideal for conversational tasks
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f" Loading tokenizer from: {MODEL_NAME}")

# Load the tokenizer — converts text ↔ token IDs
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f" Loading model from: {MODEL_NAME} (this may take a moment on first run...)")

# Load the causal language model — generates the next tokens given a prompt
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set the model to evaluation mode — disables dropout layers (not training)
model.eval()

print(" Model and tokenizer loaded successfully!")
print(f" Model type : {type(model).__name__}")
print(f" Vocab size : {tokenizer.vocab_size}")

 Loading tokenizer from: microsoft/DialoGPT-medium


C:\Users\Zaara\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Zaara\.cache\huggingface\hub\models--microsoft--DialoGPT-medium. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


 Loading model from: microsoft/DialoGPT-medium (this may take a moment on first run...)


Loading weights: 100%|█████████████████████████████████████████████████████████████| 293/293 [00:00<00:00, 8493.19it/s]


 Model and tokenizer loaded successfully!
 Model type : GPT2LMHeadModel
 Vocab size : 50257


## Step 4 — Define the Response Generation Function

In [4]:
def generate_response(user_input, chat_history_ids=None):
    """
    Generates a chatbot response for the given user input.
    Args:
        user_input (str): The message typed by the user.
        chat_history_ids: Tensor holding the full conversation history (None at the start of a new conversation).
    Returns:
        response (str): The chatbot's generated reply.
        chat_history_ids: Updated conversation history tensor.
    """
    # Tokenize the new user input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token,return_tensors='pt')

    # Append new input to conversation history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response using the model
    with torch.no_grad(): 
        chat_history_ids = model.generate( bot_input_ids, max_length=1000, pad_token_id=tokenizer.eos_token_id, no_repeat_ngram_size=3, do_sample=True,
                                           top_k=50, top_p=0.9, temperature=0.7
                                         )

    # Decode the newly generated tokens
    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0],skip_special_tokens=True
                               )

    return response, chat_history_ids
    
print(" Response generation function defined successfully!")

 Response generation function defined successfully!


## Step 5 — Run the Interactive Chatbot

In [5]:
#   SIMULATED MULTI-TURN CONVERSATION DEMO

# Predefined list of user messages to simulate a real conversation
# The last entry 'exit' tests the exit condition
demo_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "What are Transformers in NLP?",
    "Tell me something interesting about data science.",
    "Thank you!",
    "exit"
]

# Initialise conversation history
chat_history_ids = None

print("=" * 60)
print(" AI CHATBOT — DialoGPT-medium")
print(" Powered by Hugging Face Transformers")
print("=" * 60)
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
print("-" * 60)

# Iterate over each simulated user message
for user_message in demo_inputs:
    print(f"You : {user_message}")
    
    # Exit Condition
    if user_message.strip().lower() in ["exit", "quit"]:
        print("Chatbot: It was great talking to you! Goodbye!")
        print("=" * 60)
        print("Session ended...")
        break

    # Generate Response
    response, chat_history_ids = generate_response(user_message, chat_history_ids)

    # Display Response
    print(f"Chatbot: {response}")
    print("-" * 60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 AI CHATBOT — DialoGPT-medium
 Powered by Hugging Face Transformers
Chatbot: Hello! I am your AI assistant. How can I help you today?
------------------------------------------------------------
You : Hello
Chatbot: Hey, nice to see you here
------------------------------------------------------------
You : What is Artificial Intelligence?
Chatbot: You're not a bot!
------------------------------------------------------------
You : Who created Python?
Chatbot: I did. I just forgot to tell you.
------------------------------------------------------------
You : What are Transformers in NLP?
Chatbot: The one that makes you a robot
------------------------------------------------------------
You : Tell me something interesting about data science.
Chatbot: You know it's not true.
------------------------------------------------------------
You : Thank you!
Chatbot: No problem
------------------------------------------------------------
You : exit
Chatbot: It was great talking to you! Goodby

## Step 6 — Live Interactive Chatbot Loop (Run in Terminal / VS Code)

In [ ]:
#   LIVE INTERACTIVE CHATBOT — Uses input() for real dialogue

def run_chatbot():
    # Initialise empty conversation history
    chat_history_ids = None

    print("=" * 60)
    print("AI CHATBOT — DialoGPT-medium")
    print("Powered by Hugging Face Transformers")
    print("Type 'exit' or 'quit' to end the session")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 60)

    # Continuous conversation loop
    while True:
        # Accept user input
        user_input = input("You    : ").strip()
        # Validate input — skip empty messages 
        if not user_input:
            print("Chatbot: Please type something! I'm here to help.")
            print("-" * 60)
            continue

        # Exit Condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: It was great talking to you! Goodbye!")
            print("=" * 60)
            print("Session ended...")
            break

        # Generate and display chatbot response
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        print(f"Chatbot: {response}")
        print("-" * 60)

run_chatbot()

print("Live chatbot function defined. Call run_chatbot() to start a live session.")

AI CHATBOT — DialoGPT-medium
Powered by Hugging Face Transformers
Type 'exit' or 'quit' to end the session
Chatbot: Hello! I am your AI assistant. How can I help you today?
------------------------------------------------------------


You    :  Hi, whats the time?


Chatbot: I think I will be available around 8p m EST.
------------------------------------------------------------
